# AI-Task7: Job Posting Extractor (LangChain + Pydantic)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaweriamushtaq41-coder/Smart_District_2/blob/main/AI-Task7/AI_Task7_Job_Posting_Extractor.ipynb)

**Kya banana hai:** Job posting text se structured record nikalna (title, company, skills, experience level, salary range, remote status). `PydanticOutputParser` + `get_format_instructions()`, galat output `OutputFixingParser` se theek, aakhir mein ek clean JSON file.

**Concepts:** Pydantic v2, PydanticOutputParser, get_format_instructions, OutputFixingParser

**Colab mein kaise chalayen:**
1. Upar *Open In Colab* dabao (ya ye file Colab mein upload karo).
2. *Runtime > Run all* dabao.
3. Keys ho to Colab ke left side ke **Secrets** (chaabi ka icon) mein daalo. Bina key ke bhi chalta hai (**demo mode**, nakli jawab).

> Notebook ko share karne se pehle apni keys kabhi output / cell mein mat chhodo.

## 1. Libraries install aur keys (`.env`)

In [1]:
!pip -q install -U python-dotenv langchain-core langchain-openai langchain-anthropic langchain-google-genai langchain-groq langchain-classic pydantic

In [2]:
import os

# .env file banao. Keys do tareeqon se aa sakti hain:
#  1) Colab "Secrets" (left side ka chaabi wala icon), secret ka naam wahi jo neeche list mein hai, "Notebook access" ON
#  2) apni .env file Colab ke "Files" panel (left side ka folder icon) mein drag-drop kar do, run karne se pehle
NAMES = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY"]

existing = {}                       # pehle se .env ho (upload ki hui) to uski keys bacha lo
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if "=" in line and not line.startswith("#"):
                key, value = line.strip().split("=", 1)
                existing[key] = value

lines = []
for name in NAMES:
    value = existing.get(name, "")
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:          # Colab nahi hai, ya ye secret nahi banaya
        pass
    lines.append(f"{name}={value}")

with open(".env", "w") as f:
    f.write("\n".join(lines) + "\n")

from dotenv import load_dotenv
load_dotenv(".env", override=True)

found = [n for n in NAMES if os.getenv(n)]
print("Keys mili:", found if found else "koi nahi (demo mode chalega)")

Keys mili: koi nahi (demo mode chalega)


## 2. Model factory
`get_chat_model("groq")` jaisa call asli LangChain model deta hai (ChatOpenAI, ChatAnthropic, ChatGoogleGenerativeAI, ChatGroq).
Provider badalna ho to sirf naam badlo, chain wahi rehti hai. `demo_model` / `demo_stream_model` key ke baghair nakli model dete hain jo chain mein asli ki tarah lagte hain.

In [3]:
import os

from dotenv import load_dotenv
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda

load_dotenv()

PROVIDERS = ["openai", "claude", "gemini", "groq"]

KEY_NAMES = {
    "openai": "OPENAI_API_KEY",
    "claude": "ANTHROPIC_API_KEY",
    "gemini": "GOOGLE_API_KEY",
    "groq": "GROQ_API_KEY",
}

MODELS = {
    "openai": os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    "claude": os.getenv("ANTHROPIC_MODEL", "claude-3-5-haiku-latest"),
    "gemini": os.getenv("GEMINI_MODEL", "gemini-1.5-flash"),
    "groq": os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"),
}


def available_providers():
    """Jin providers ki key .env mein hai."""
    return [p for p in PROVIDERS if os.getenv(KEY_NAMES[p], "").strip()]


def get_chat_model(provider, temperature=0.3):
    """Asli LangChain model wapas deta hai. Import andar hai taake sirf wahi package chahiye jo use ho."""
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODELS["openai"], temperature=temperature)
    if provider == "claude":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=MODELS["claude"], temperature=temperature)
    if provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=MODELS["gemini"], temperature=temperature)
    if provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=MODELS["groq"], temperature=temperature)
    raise ValueError(f"Unknown provider: {provider}")


# ---------------- demo (nakli) models ----------------
def demo_stream_model(text, delay=0.004):
    """Ek fixed jawab deta hai, harf harf stream hota hai (asli streaming jaisa)."""
    return FakeListChatModel(responses=[text], sleep=delay)


def demo_model(responder):
    """responder(prompt_text) -> jawab ka text. Prompt dekh kar jawab chunta hai,
    is liye RunnableParallel mein (jahan tarteeb pakki nahi) bhi theek chalta hai."""
    def run(prompt_value):
        text = prompt_value if isinstance(prompt_value, str) else prompt_value.to_string()
        return AIMessage(content=responder(text))
    return RunnableLambda(run)

## 3. Schema (Pydantic model)
Experience level `Literal` hai, salary positive honi chahiye, `salary_max >= salary_min`.

In [4]:
from typing import Literal, Optional

from pydantic import BaseModel, Field, RootModel, field_validator, model_validator

ExperienceLevel = Literal["Intern", "Junior", "Mid", "Senior", "Lead"]
RemoteStatus = Literal["Remote", "Hybrid", "On-site"]


class JobPosting(BaseModel):
    job_title: str = Field(description="Job ka title")
    company: str
    required_skills: list[str] = Field(min_length=1, description="Zaroori skills ki list")
    experience_level: ExperienceLevel
    salary_min: Optional[float] = Field(default=None, description="Mahana tankhwah (kam se kam), PKR")
    salary_max: Optional[float] = Field(default=None, description="Mahana tankhwah (zyada se zyada), PKR")
    remote_status: RemoteStatus

    @field_validator("experience_level", mode="before")
    @classmethod
    def clean_level(cls, value):
        # "senior " / "SENIOR" ko "Senior" bana do. "Mid-level" jaisi cheez phir bhi ghalat rahegi (Literal ki wajah se).
        return value.strip().title() if isinstance(value, str) else value

    @field_validator("salary_min", "salary_max")
    @classmethod
    def salary_positive(cls, value):
        if value is not None and value <= 0:
            raise ValueError("salary positive honi chahiye")
        return value

    @field_validator("required_skills")
    @classmethod
    def clean_skills(cls, skills):
        return [s.strip() for s in skills if s.strip()]

    @model_validator(mode="after")
    def check_range(self):
        if self.salary_min and self.salary_max and self.salary_max < self.salary_min:
            raise ValueError("salary_max, salary_min se chhoti nahi ho sakti")
        return self


class JobPostingList(RootModel[list[JobPosting]]):
    """Saare nikale hue records ek file mein likhne ke liye."""

## 4. Job postings (4 text files ka text)

In [5]:
POSTINGS = {
    "posting1.txt": """Junior Python Developer
Company: Brightpath Software
Location: Narowal (Hybrid)
Experience: 1-2 years
Salary: PKR 120,000 - 160,000 per month

Brightpath Software is hiring a junior developer to build internal tools and REST APIs.
Skills: Python, Django, SQL, Git.""",
    "posting2.txt": """Data Analyst
Company: Greenfield Analytics
Location: Lahore (Remote)
Experience: 3-5 years
Salary: PKR 250,000 - 320,000 per month

You will build dashboards and clean large datasets for retail clients.
Skills: SQL, Excel, Power BI, Python.""",
    "posting3.txt": """Senior Backend Engineer
Company: Craftly Labs
Location: Islamabad (On-site)
Experience: 6+ years
Salary: PKR 500,000 - 650,000 per month

Lead the design of our payment services and mentor a small team.
Skills: Node.js, PostgreSQL, Docker, AWS, System Design.""",
    "posting4.txt": """Frontend Intern
Company: Pixel Studio
Location: Sialkot (Remote)
Experience: 0 years (students welcome)

Help our designers turn mock-ups into React pages. Stipend discussed in interview.
Skills: HTML, CSS, JavaScript, React.""",
}

## 5. Extraction ka pipeline
1. `parser.get_format_instructions()` prompt mein dal dete hain, taake model ko JSON ki shakal pata ho.
2. Jawab `parser.parse()` se Pydantic object banta hai.
3. Parse fail ho to `OutputFixingParser` galat jawab model ko wapas bhej kar theek karwata hai.

In [6]:
import json
import os
import re

from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

try:                      # naye LangChain mein OutputFixingParser langchain-classic mein hai
    from langchain.output_parsers import OutputFixingParser
except Exception:
    from langchain_classic.output_parsers import OutputFixingParser

In [7]:
parser = PydanticOutputParser(pydantic_object=JobPosting)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You extract structured data from job postings. "
               "Reply with JSON only, nothing else.\n{format_instructions}"),
    ("human", "Job posting:\n\"\"\"\n{text}\n\"\"\""),
]).partial(format_instructions=parser.get_format_instructions())     # format ki hidayat khud bhar di


# ---------------- demo (nakli model) ----------------
def demo_extract(text):
    """Demo ke liye simple rules se wahi cheezein nikalta hai jo asli model nikalta."""
    lines = [l.strip() for l in text.strip().splitlines() if l.strip() and l.strip() != '"""']
    company = re.search(r"Company:\s*(.+)", text).group(1).strip()
    remote = re.search(r"\((Remote|Hybrid|On-site)\)", text).group(1)
    skills = [s.strip().rstrip(".") for s in re.search(r"Skills:\s*(.+)", text).group(1).split(",")]
    years = int(re.search(r"Experience:\s*(\d+)", text).group(1))
    level = "Intern" if years == 0 else "Junior" if years <= 2 else "Mid" if years <= 5 else "Senior"
    salary = re.search(r"PKR\s*([\d,]+)\s*-\s*([\d,]+)", text)
    low = float(salary.group(1).replace(",", "")) if salary else None
    high = float(salary.group(2).replace(",", "")) if salary else None
    return {"job_title": lines[0], "company": company, "required_skills": skills,
            "experience_level": level, "salary_min": low, "salary_max": high, "remote_status": remote}


def demo_brain(prompt_text):
    """Nakli model ka dimaagh. Pehle jawab mein jaan boojh kar ek ghalti karta hai (Data Analyst)
    taake OutputFixingParser ka kaam nazar aaye."""
    if "did not satisfy the constraints" in prompt_text:                   # OutputFixingParser ka sawal
        bad_part = prompt_text.split("Completion:")[1].split("Above, the Completion")[0]
        broken = re.search(r"\{.*\}", bad_part, flags=re.DOTALL).group(0)
        data = json.loads(broken)
        data["experience_level"] = "Mid"
        return json.dumps(data)
    data = demo_extract(prompt_text.split("Job posting:")[1])
    if data["job_title"] == "Data Analyst":
        data["experience_level"] = "Mid-level"                            # ghalat: Literal mein nahi
        return "Sure! Here is the JSON you asked for:\n" + json.dumps(data)
    return json.dumps(data)


def extract_one(text, llm):
    """(record, tareeqa) wapas deta hai. Pehle seedha parse, na ho to OutputFixingParser."""
    raw = (prompt | llm | StrOutputParser()).invoke({"text": text})
    try:
        return parser.parse(raw), "seedha sahi"
    except OutputParserException:
        fixer = OutputFixingParser.from_llm(parser=parser, llm=llm)
        return fixer.parse(raw), "OutputFixingParser ne theek kiya"

## 6. Provider chuno

In [8]:
FORCE_DEMO = False      # True likhoge to hamesha nakli model
PROVIDER = None         # "groq", "openai", "claude" ya "gemini". None = jis ki key mili
demo = FORCE_DEMO
provider = PROVIDER or (available_providers() or [None])[0]
if provider is None:
    print("[!] Koi API key nahi mili, DEMO mode chal raha hai.")
    demo, provider = True, "groq"
print("provider =", provider, "| demo =", demo)

[!] Koi API key nahi mili, DEMO mode chal raha hai.
provider = groq | demo = True


## 7. Chalao

In [9]:
llm = demo_model(demo_brain) if demo else get_chat_model(provider, temperature=0)

records, problems = [], []
for name, text in POSTINGS.items():
    try:
        record, how = extract_one(text, llm)
        records.append(record)
        print(f"OK   {name:<14} {record.job_title:<26} [{how}]")
    except Exception as e:               # ek file kharab ho to baaki chalti rahen
        problems.append((name, str(e)[:200]))
        print(f"FAIL {name:<14} {str(e)[:80]}")

os.makedirs("output", exist_ok=True)
with open("output/job_postings.json", "w", encoding="utf-8") as f:
    f.write(JobPostingList(records).model_dump_json(indent=2))
print(f"\n{len(records)} sahi, {len(problems)} fail. Output: output/job_postings.json")

OK   posting1.txt   Junior Python Developer    [seedha sahi]
OK   posting2.txt   Data Analyst               [OutputFixingParser ne theek kiya]
OK   posting3.txt   Senior Backend Engineer    [seedha sahi]
OK   posting4.txt   Frontend Intern            [seedha sahi]

4 sahi, 0 fail. Output: output/job_postings.json


## 8. Natija dekho

In [10]:
print(open("output/job_postings.json").read()[:1200])

[
  {
    "job_title": "Junior Python Developer",
    "company": "Brightpath Software",
    "required_skills": [
      "Python",
      "Django",
      "SQL",
      "Git"
    ],
    "experience_level": "Junior",
    "salary_min": 120000.0,
    "salary_max": 160000.0,
    "remote_status": "Hybrid"
  },
  {
    "job_title": "Data Analyst",
    "company": "Greenfield Analytics",
    "required_skills": [
      "SQL",
      "Excel",
      "Power BI",
      "Python"
    ],
    "experience_level": "Mid",
    "salary_min": 250000.0,
    "salary_max": 320000.0,
    "remote_status": "Remote"
  },
  {
    "job_title": "Senior Backend Engineer",
    "company": "Craftly Labs",
    "required_skills": [
      "Node.js",
      "PostgreSQL",
      "Docker",
      "AWS",
      "System Design"
    ],
    "experience_level": "Senior",
    "salary_min": 500000.0,
    "salary_max": 650000.0,
    "remote_status": "On-site"
  },
  {
    "job_title": "Frontend Intern",
    "company": "Pixel Studio",
    "requir

## 9. Format instructions kaisi dikhti hain?
Yehi text model ko prompt mein jata hai.

In [11]:
print(parser.get_format_instructions()[:900])

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"job_title": {"description": "Job ka title", "title": "Job Title", "type": "string"}, "company": {"title": "Company", "type": "string"}, "required_skills": {"description": "Zaroori skills ki list", "items": {"type": "string"}, "minItems": 1, "title": "Required Skills", "type": "array"}, "experience_level": {"enum": ["Intern", "Junior", "Mid", "Senior", "Lead"], "title": "Experience Level", "type": "string"}, "salary_min": {"anyOf


## Kya seekha
- LLM ka jawab text hota hai, production mein structure chahiye. Isliye Pydantic schema + parser.
- `get_format_instructions()` model ko batata hai JSON kaisa chahiye.
- `OutputFixingParser` galat output (jaise `"Mid-level"` jo Literal mein nahi) theek karwata hai.
- Demo mein Data Analyst posting ka pehla jawab jaan boojh kar ghalat hota hai taake fixing parser ka kaam nazar aaye.